# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

**Dataset DOI**: [10.71728/senscience.vcs2-05nj](https://doi.org/10.71728/senscience.vcs2-05nj)

**Description**: The dataset contains survey data on mental health indicators among residents of Kilifi County. It includes demographic details and measurements of psychological symptoms along with scores from assessments such as GAD-7, PHQ-9, and PSQ.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
# Print name and description for sanity check
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Let's retrieve all record sets in the dataset and inspect their IDs and attached field/column `@id`s.

In [ ]:
print("Record sets in the dataset:")
for rs in metadata.record_sets:
    print(f"  Record set: @id={rs.id}, name={rs.name}")
    print("    Fields/columns:")
    for field in rs.fields:
        print(f"      @id={field.id}, name={field.name}")
        if hasattr(field, 'data_type'):
            print(f"        data_type: {field.data_type}")
    print()

## 3. Data Extraction
Load data from each main record set into a DataFrame for analysis.

We reference each by its Croissant `@id`. If the set is large, we show the first few records and column names for inspection.

In [ ]:
# Gather list of record set @ids
record_sets = [rs.id for rs in metadata.record_sets]

dataframes = {}
for record_set_id in record_sets:
    print(f"\nLoading record set '@id': {record_set_id}")
    
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns: {df.columns.tolist()}")
        display(df.head(3))
    except Exception as e:
        print(f"  Could not load records for record_set {record_set_id} ({e})")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

For demonstration, we find the main survey record set (e.g., participant data), pick a numeric field (e.g., PHQ-9 score), filter and normalize it, and group by a demographic attribute (e.g., gender or education).

_**Note**: All references use the Croissant `@id` fields._

In [ ]:
# Select a main record set by @id (e.g., the survey responses table)
# Replace this with the correct @id found previously, e.g., "cr:RecordSet/SurveyResponses"
survey_record_set_id = record_sets[0] # adjust if multiple record sets
df = dataframes[survey_record_set_id]

# Find a numeric field (e.g., PHQ-9 total score) and a group field (e.g., gender)
# List columns for reference:
print(f"Columns: {df.columns.tolist()}")

# Guess likely field IDs (example):
numeric_field_id = None
for col in df.columns:
    if "phq" in str(col).lower() and ("score" in str(col).lower() or str(col).endswith("_total")):
        numeric_field_id = col
        break

if numeric_field_id is None:
    # fallback to just the first numeric-looking column
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break

print(f"Using numeric field: {numeric_field_id}")

# Choose a common grouping field, e.g., 'gender' or 'level_of_education'
group_field_id = None
for gname in ['gender', 'sex', 'level_of_education', 'marital_status']:
    for col in df.columns:
        if gname in str(col).lower():
            group_field_id = col
            break
    if group_field_id:
        break

print(f"Using group field: {group_field_id}")

if numeric_field_id is not None:
    threshold = df[numeric_field_id].mean() # Just as an example, filter above mean
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (show first 5):")
    display(filtered_df.head())

    # Normalize field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id} for filtered records (show first 5):")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by the field
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame('mean').join(
            filtered_df.groupby(group_field_id)[numeric_field_id].count().to_frame('n'))
        print(f"Grouped data by {group_field_id} (mean and count):")
        display(grouped_df)
else:
    print("No numeric field found for analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll plot the distribution of the chosen numeric field and the group-wise means if possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None:
    # Distribution
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=10)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()

    # Boxplot by group field
    if group_field_id is not None and group_field_id in df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=30)
        plt.show()
else:
    print("No numeric field found for visualization.")

## 6. Conclusion
We have loaded, examined, and explored the *A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya* using the Croissant schema and the `mlcroissant` Python library. Summary statistics and visualizations can inform further public health and machine learning analyses. For deeper exploration, consult the field and record set definitions accessible using their Croissant `@id` references and refer to the dataset license and documentation.